# Forum Diskusi 3 - Pengantar Data Science
## Pertemuan 3: Data Cleaning, Missing Values, Outlier, dan Ekstraksi Data

Nama: Difa Asmana  
NIM: 240401010008

In [14]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import requests
from pandas import json_normalize

In [15]:
df = pd.read_csv('housing_dirty.csv')
print("Shape awal:", df.shape)
df.head()

Shape awal: (130, 7)


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang


## Eksplorasi Awal
Pada bagian ini saya mengecek informasi dasar dataset, statistik deskriptif, dan jumlah missing values.

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB


In [17]:
df.describe()

,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


In [18]:
df.isnull().sum()

,0
id,0
luas_m2,18
harga_juta,17
kota,0
kamar,10
tahun_bangun,0
kondisi,0


## Hapus Duplikat
Pada bagian ini saya menghapus baris data yang duplikat.

In [19]:
print("Jumlah duplikat sebelum dihapus:", df.duplicated().sum())
df.drop_duplicates(inplace=True)
print("Jumlah duplikat setelah dihapus:", df.duplicated().sum())
print("Shape setelah hapus duplikat:", df.shape)

Jumlah duplikat sebelum dihapus: 0
Jumlah duplikat setelah dihapus: 0
Shape setelah hapus duplikat: (130, 7)


## Normalisasi String
Pada bagian ini saya merapikan penulisan kolom kota dan kondisi agar konsisten.

In [20]:
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

print(df[['kota', 'kondisi']].head())

    kota kondisi
0  Jogja    baik
1  Medan   bagus
2  Depok    baik
3    Ygy    baik
4  Medan  sedang


## Imputasi Missing Values
Pada bagian ini saya mengisi nilai yang hilang menggunakan median untuk kolom numerik dan modus untuk kolom kategorik.

In [21]:
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

df.isnull().sum()

,0
id,0
luas_m2,0
harga_juta,0
kota,0
kamar,0
tahun_bangun,0
kondisi,0


## Menangani Outlier dengan IQR Fence
Pada bagian ini saya membatasi nilai ekstrem menggunakan metode IQR pada kolom harga_juta, luas_m2, dan tahun_bangun.

In [22]:
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)

df[['harga_juta', 'luas_m2', 'tahun_bangun']].describe()

,harga_juta,luas_m2,tahun_bangun
count,130.000000,130.000000,130.000000
mean,686.490385,188.274423,2001.542308
std,404.633957,95.297150,13.505818
min,-422.750000,-50.000000,1960.500000
25%,380.500000,101.600000,1991.250000
50%,655.000000,193.800000,2002.000000
75%,916.000000,266.150000,2011.750000
max,1719.250000,512.975000,2042.500000


## Validasi Akhir
Pada bagian ini saya memastikan tidak ada missing values dan duplikat yang tersisa.

In [23]:
print("Total missing values akhir:", df.isnull().sum().sum())
print("Total duplikat akhir:", df.duplicated().sum())
print("Shape akhir:", df.shape)

Total missing values akhir: 0
Total duplikat akhir: 0
Shape akhir: (130, 7)


In [24]:
assert df.isnull().sum().sum() == 0, "Masih ada missing!"
assert df.duplicated().sum() == 0, "Masih ada duplikat!"
print("Validasi berhasil, data sudah bersih.")

Validasi berhasil, data sudah bersih.


## Ekspor Dataset Bersih
Pada bagian ini saya menyimpan dataset yang sudah dibersihkan menjadi file CSV baru.

In [25]:
df.to_csv('housing_clean.csv', index=False)
print("File housing_clean.csv berhasil dibuat.")

File housing_clean.csv berhasil dibuat.


## Ekstraksi Data dari REST API Publik
Pada bagian ini saya mengambil data dari JSONPlaceholder API, lalu mengubahnya menjadi DataFrame.

In [26]:
URL = "https://jsonplaceholder.typicode.com/users"
response = requests.get(URL, timeout=10)

print("Status code:", response.status_code)

if response.status_code == 200:
    data_api = response.json()
    df_api = json_normalize(data_api, sep='_')
    print(df_api[['id', 'name', 'email', 'address_city']])
else:
    print("Gagal mengambil data API")

Status code: 200
   id                      name                      email    address_city
0   1             Leanne Graham          Sincere@april.biz     Gwenborough
1   2              Ervin Howell          Shanna@melissa.tv     Wisokyburgh
2   3          Clementine Bauch         Nathan@yesenia.net   McKenziehaven
3   4          Patricia Lebsack  Julianne.OConner@kory.org     South Elvis
4   5          Chelsey Dietrich   Lucio_Hettinger@annie.ca      Roscoeview
5   6      Mrs. Dennis Schulist    Karley_Dach@jasper.info   South Christy
6   7           Kurtis Weissnat     Telly.Hoeger@billy.biz       Howemouth
7   8  Nicholas Runolfsdottir V       Sherwood@rosamond.me       Aliyaview
8   9           Glenna Reichert    Chaim_McDermott@dana.io  Bartholomebury
9  10        Clementina DuBuque     Rey.Padberg@karina.biz     Lebsackbury


## Kesimpulan
Pada praktikum ini saya mempelajari proses data cleaning mulai dari eksplorasi awal, menghapus duplikat, normalisasi string, imputasi missing values, menangani outlier, mengekspor data bersih, dan mengambil data dari REST API publik.